## Install

In [ ]:
!git clone https://github.com/Lareton/lensless_cv.git

In [ ]:
%cd lensless_cv

In [ ]:
!apt-get update -qq
!apt-get install -y -qq python3-tk
!pip install -r requirements.txt

#### здесб может понадобиться перезапустить среду выполнения в колабе - тогда нужно выполнить команду %cd lensless_cv заново

In [ ]:
import os
from getpass import getpass

use_comet = True

if use_comet:
    os.environ["COMET_API_KEY"] = getpass("COMET_API_KEY: ")
else:
    os.environ["COMET_API_KEY"] = ""

## Download custom dataset

In [ ]:
!hf download Lareton/lensless_models --local-dir saved

In [ ]:
DATASET_ZIP_URL = "PASTE_GOOGLE_DRIVE_OR_DIRECT_ZIP_URL_HERE"

ZIP_PATH = "data/custom_dataset.zip"
CUSTOM_DATA_DIR = "data/custom_dataset"
RECON_DIR = "data/demo_reconstructions"
CHECKPOINT_PATH = "saved/pre4-leadmm5-post4-scale4/best.pt"

In [ ]:
import os
import subprocess
from pathlib import Path

Path("data").mkdir(exist_ok=True)

if "drive.google.com" in DATASET_ZIP_URL:
    !pip install -q gdown
    !gdown --fuzzy "$DATASET_ZIP_URL" -O "$ZIP_PATH"
else:
    !wget -O "$ZIP_PATH" "$DATASET_ZIP_URL"

print("Downloaded to:", ZIP_PATH)

## Unzip dataset

In [ ]:
import zipfile
from pathlib import Path

custom_root = Path(CUSTOM_DATA_DIR)
custom_root.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    archive.extractall(custom_root)

print("Extracted files:")
for path in sorted(custom_root.rglob("*"))[:30]:
    print(path)

## Dataset Root

In [ ]:
from pathlib import Path

def find_dataset_root(root):
    root = Path(root)
    candidates = [root] + [p for p in root.rglob("*") if p.is_dir()]
    for candidate in candidates:
        if (candidate / "lensless").is_dir() and (candidate / "masks").is_dir():
            return candidate
    raise FileNotFoundError("Could not find dataset root with lensless/ and masks/ folders")

DATASET_ROOT = find_dataset_root(CUSTOM_DATA_DIR)
HAS_GROUND_TRUTH = (DATASET_ROOT / "lensed").is_dir()

print("Dataset root:", DATASET_ROOT)
print("Has ground truth:", HAS_GROUND_TRUTH)

## Inference

In [ ]:
writer_flag = "" if use_comet else "writer.enabled=false"

!python3 inference.py \
  datasets=custom_dir \
  datasets.test.data_dir="$DATASET_ROOT" \
  model=pre4_leadmm5_post4 \
  checkpoint="$CHECKPOINT_PATH" \
  output_dir="$RECON_DIR" \
  calculate_metrics=false \
  writer.experiment_name=demo-custom-inference \
  $writer_flag

In [ ]:
from pathlib import Path

recon_paths = sorted(Path(RECON_DIR).glob("*.png"))

print("Reconstructions:", len(recon_paths))
for path in recon_paths[:10]:
    print(path)

## Vizualization

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

lensless_dir = DATASET_ROOT / "lensless"
lensed_dir = DATASET_ROOT / "lensed"
recon_dir = Path(RECON_DIR)

image_ids = [p.stem for p in sorted(recon_dir.glob("*.png"))[:5]]

cols = 3 if HAS_GROUND_TRUTH else 2
fig, axes = plt.subplots(len(image_ids), cols, figsize=(5 * cols, 4 * len(image_ids)))

if len(image_ids) == 1:
    axes = [axes]

for row, image_id in enumerate(image_ids):
    current_axes = axes[row] if len(image_ids) > 1 else axes

    col = 0
    if HAS_GROUND_TRUTH:
        gt_path = next(lensed_dir.glob(f"{image_id}.*"))
        current_axes[col].imshow(Image.open(gt_path))
        current_axes[col].set_title("ground truth")
        current_axes[col].axis("off")
        col += 1

    lensless_path = next(lensless_dir.glob(f"{image_id}.*"))
    current_axes[col].imshow(Image.open(lensless_path))
    current_axes[col].set_title("lensless")
    current_axes[col].axis("off")
    col += 1

    current_axes[col].imshow(Image.open(recon_dir / f"{image_id}.png"))
    current_axes[col].set_title("reconstruction")
    current_axes[col].axis("off")

plt.tight_layout()
plt.show()

## Metrics

In [ ]:
if HAS_GROUND_TRUTH:
    !python3 calculate_metrics.py \
      ground_truth_dir="$DATASET_ROOT/lensed" \
      reconstruction_dir="$RECON_DIR" \
      writer.experiment_name=demo-custom-metrics \
      $writer_flag
else:
    print("No lensed/ folder found, metrics are skipped.")

In [ ]:
## Digicam

In [ ]:
!python3 download_dataset.py output_dir=data/digicam split=test

In [ ]:
!python3 inference.py \
  model=pre4_leadmm5_post4 \
  checkpoint="$CHECKPOINT_PATH" \
  output_dir=data/demo_digicam_test \
  writer.experiment_name=demo-digicam-test \
  $writer_flag

In [ ]:
!python3 benchmark.py \
  model=pre4_leadmm5_post4 \
  checkpoint="$CHECKPOINT_PATH" \
  writer.experiment_name=demo-benchmark \
  $writer_flag